# HST Gravitational Lensing Dataset Generator
**100,000 synthetic HST-like lensing images for CNN training**

| Class | Count | Notes |
|---|---|---|
| `no_lens` | 50,000 | Unlensed galaxy fields |
| `lens_no_subhalo` | 25,000 | Clean Einstein rings / arcs |
| `lens_with_subhalo` | 25,000 | Rings / arcs + subtle NFW subhalo |

| Realism | Fraction |
|---|---|
| `clean` | 20% |
| `semi_messy` | 30% |
| `messy` | 50% |

**Models supported:**
- **Model 1** → `lens_label`: lens / no_lens  
- **Model 2** → `lens_type`: ring / arc / double / quad / partial_ring  
- **Model 3** → `subhalo_label`: subhalo / no_subhalo


## 📦 Cell 1 — Install dependencies

In [ ]:
# ── Install lenstronomy and dependencies ──────────────────────────────────
# Takes ~3-4 minutes on Colab T4
!pip install lenstronomy==1.11.8 pillow tqdm scipy numpy pandas matplotlib -q
print('✅ Dependencies installed')

## 📁 Cell 2 — Upload pipeline files

In [ ]:
# Option A: clone from your GitHub repo
# !git clone https://github.com/YOUR_USERNAME/hst-lensing-pipeline.git
# %cd hst-lensing-pipeline

# Option B: upload the zip and extract
# from google.colab import files
# uploaded = files.upload()   # upload hst_pipeline.zip
# !unzip -q hst_pipeline.zip

# Option C: mount Google Drive (recommended for large datasets)
from google.colab import drive
drive.mount('/content/drive')

import sys, os
# Point to wherever you placed the pipeline files
PIPELINE_DIR = '/content/hst_pipeline'   # adjust if needed
sys.path.insert(0, PIPELINE_DIR)

print('Pipeline dir:', PIPELINE_DIR)
print('Files found:', os.listdir(PIPELINE_DIR))

## ⚙️ Cell 3 — Configure dataset output path

In [ ]:
# ── Override the output path in config.py if needed ──────────────────────
# By default ROOT_DIR = /content/hst_lensing_dataset
# If you want to save to Google Drive:
#   ROOT_DIR = '/content/drive/MyDrive/hst_lensing_dataset'

import config
# Optionally override:
# config.ROOT_DIR     = '/content/drive/MyDrive/hst_lensing_dataset'
# config.IMAGES_DIR   = config.ROOT_DIR + '/images'
# config.METADATA_PATH= config.ROOT_DIR + '/metadata.csv'
# config.SPLITS_DIR   = config.ROOT_DIR + '/splits'
# config.MODEL2_DIR   = config.ROOT_DIR + '/model_2_lens_type'

print('Output root:', config.ROOT_DIR)
print('Total images to generate:', config.TOTAL_IMAGES)

## 🚀 Cell 4 — Generate the full dataset

In [ ]:
from generators.batch_runner import run_pipeline

# ── Generation settings ───────────────────────────────────────────────────
# num_workers=1  → sequential (safer in Colab, avoids CUDA fork issues)
# num_workers=4  → parallel (faster, use if no CUDA multiprocessing errors)
# resume=True    → skip already-generated images (safe to re-run)

run_pipeline(
    num_workers = 1,    # increase to 4 if Colab allows
    resume      = True,
)

# ─── EXPECTED TIME ON COLAB T4 ───────────────────────────────────────────
# lenstronomy is CPU-bound. Approximate times:
#   1 worker  → ~5-8 hours for 100k images
#   4 workers → ~1.5-2.5 hours
# Tip: run with num_workers=4 then re-run with resume=True if it times out.

## 🔍 Cell 5 — Quick generation test (small batch)

In [ ]:
# Test with 30 images first to verify physics is working
import config as cfg

# Temporarily reduce counts
_orig = dict(cfg.CLASS_COUNTS)
cfg.CLASS_COUNTS['no_lens']            = 12
cfg.CLASS_COUNTS['lens_no_subhalo']    = 9
cfg.CLASS_COUNTS['lens_with_subhalo']  = 9

import importlib, generators.batch_runner as br
importlib.reload(br)

TEST_DIR = '/content/hst_test_30'
cfg.ROOT_DIR      = TEST_DIR
cfg.IMAGES_DIR    = TEST_DIR + '/images'
cfg.METADATA_PATH = TEST_DIR + '/metadata.csv'
cfg.SPLITS_DIR    = TEST_DIR + '/splits'
cfg.MODEL2_DIR    = TEST_DIR + '/model_2_lens_type'

br.run_pipeline(num_workers=1, resume=False)

# Restore originals
cfg.CLASS_COUNTS.update(_orig)
print('Test run complete!')

## 🖼️ Cell 6 — Visualize sample images

In [ ]:
%matplotlib inline
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from utils.visualize import (
    plot_sample_grid,
    plot_arc_morphologies,
    plot_class_breakdown,
    plot_physics_distributions,
    plot_subhalo_comparison,
)
import config
import os

META  = os.path.join(config.ROOT_DIR, 'metadata.csv')
IMGS  = config.IMAGES_DIR
FIGS  = os.path.join(config.ROOT_DIR, 'figures')
os.makedirs(FIGS, exist_ok=True)

# Grid: all classes × all realism levels
plot_sample_grid(
    metadata_csv=META, images_dir=IMGS,
    n_per_cell=4,
    save_path=os.path.join(FIGS, 'sample_grid.png')
)

In [ ]:
# Morphology samples
plot_arc_morphologies(
    metadata_csv=META, images_dir=IMGS,
    n_per_type=5, realism='messy',
    save_path=os.path.join(FIGS, 'arc_morphologies.png')
)

In [ ]:
# Class composition bars
plot_class_breakdown(
    metadata_csv=META,
    save_path=os.path.join(FIGS, 'class_breakdown.png')
)

In [ ]:
# Physics parameter histograms
plot_physics_distributions(
    metadata_csv=META,
    save_path=os.path.join(FIGS, 'physics_distributions.png')
)

In [ ]:
# Subhalo comparison (should look nearly identical — subtle perturbation)
plot_subhalo_comparison(
    metadata_csv=META, images_dir=IMGS,
    n_pairs=5, realism='messy',
    save_path=os.path.join(FIGS, 'subhalo_comparison.png')
)

## 📊 Cell 7 — Metadata inspection

In [ ]:
import pandas as pd, config
df = pd.read_csv(config.METADATA_PATH)

print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
display(df.head(5))

print('\n── Class × Realism crosstab ──')
display(pd.crosstab(df['main_class'], df['realism']))

print('\n── Lens type distribution ──')
display(df[df['lens_label']=='lens']['lens_type'].value_counts())

print('\n── Split distribution ──')
display(df['split'].value_counts())

## 🤖 Cell 8 — Load dataset for CNN training

In [ ]:
# Install PyTorch (already available in Colab, but pin versions if needed)
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
from utils.dataset_loader import make_dataloaders, sanity_check
import config

# ── Stage 1: Model 1 — lens detection, clean only ─────────────────────────
loaders_m1_s1 = make_dataloaders(
    model_name  = 'model1',
    stage       = 1,         # clean only
    batch_size  = 64,
    num_workers = 2,
    root_dir    = config.ROOT_DIR,
)

print('Model 1 Stage 1 train batches:', len(loaders_m1_s1['train']))
print('Model 1 Stage 1 val   batches:', len(loaders_m1_s1['val']))

# Peek at one batch
imgs, labels = next(iter(loaders_m1_s1['train']))
print(f'Batch shape: {imgs.shape} | Labels: {labels[:8].tolist()}')

In [ ]:
# ── Stage 2: Model 1 — all realism levels ─────────────────────────────────
loaders_m1_s2 = make_dataloaders(
    model_name = 'model1',
    stage      = 2,          # clean + semi_messy + messy
    batch_size = 64,
    num_workers= 2,
    root_dir   = config.ROOT_DIR,
)
print('Model 1 Stage 2 train batches:', len(loaders_m1_s2['train']))

# ── Model 2: lens type (morphology) ───────────────────────────────────────
loaders_m2 = make_dataloaders(
    model_name = 'model2',
    stage      = 2,
    batch_size = 64,
    num_workers= 2,
    root_dir   = config.ROOT_DIR,
)
print('Model 2 train batches:', len(loaders_m2['train']))

# ── Model 3: subhalo detection ─────────────────────────────────────────────
loaders_m3 = make_dataloaders(
    model_name = 'model3',
    stage      = 2,
    batch_size = 64,
    num_workers= 2,
    root_dir   = config.ROOT_DIR,
)
print('Model 3 train batches:', len(loaders_m3['train']))

## 🏗️ Cell 9 — Example CNN (ResNet-18 baseline)
Plug-and-play baseline for any of the 3 models.

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

def make_resnet18(num_classes: int, grayscale: bool = True) -> nn.Module:
    """
    ResNet-18 adapted for 1-channel (grayscale) HST images.
    First conv layer is replaced to accept 1 input channel.
    """
    model = models.resnet18(weights=None)
    if grayscale:
        # Replace conv1: 3→1 channel, keep kernel/stride/padding
        model.conv1 = nn.Conv2d(
            1, 64, kernel_size=7, stride=2, padding=3, bias=False
        )
    # Replace final FC for our number of classes
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Instantiate for Model 1 (binary) ─────────────────────────────────────
model_1 = make_resnet18(num_classes=2).to(device)
print('Model 1 parameters:', sum(p.numel() for p in model_1.parameters())/1e6, 'M')

# ── Instantiate for Model 2 (5-class morphology) ─────────────────────────
model_2 = make_resnet18(num_classes=5).to(device)

# ── Instantiate for Model 3 (binary subhalo) ─────────────────────────────
model_3 = make_resnet18(num_classes=2).to(device)

print('✅ Models ready')

In [ ]:
# ── Simple training loop skeleton ─────────────────────────────────────────
# (Replace with your actual training code / Lightning / HuggingFace Trainer)

from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(imgs)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += len(imgs)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss   = criterion(logits, labels)
        total_loss += loss.item() * len(imgs)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += len(imgs)
    return total_loss / total, correct / total


# ── Stage 1 training — Model 1 on clean data ──────────────────────────────
# (Uncomment and run when ready)

# EPOCHS_STAGE1 = 30
# optimizer  = Adam(model_1.parameters(), lr=1e-3, weight_decay=1e-4)
# scheduler  = CosineAnnealingLR(optimizer, T_max=EPOCHS_STAGE1)
# criterion  = nn.CrossEntropyLoss(
#     weight=loaders_m1_s1['train'].dataset.get_class_weights().to(device)
# )
#
# for epoch in range(EPOCHS_STAGE1):
#     tr_loss, tr_acc = train_one_epoch(model_1, loaders_m1_s1['train'],
#                                        optimizer, criterion, device)
#     vl_loss, vl_acc = evaluate(model_1, loaders_m1_s1['val'],
#                                 criterion, device)
#     scheduler.step()
#     print(f'Ep {epoch+1:3d}/{EPOCHS_STAGE1} | '
#           f'tr_loss={tr_loss:.4f} tr_acc={tr_acc:.4f} | '
#           f'val_loss={vl_loss:.4f} val_acc={vl_acc:.4f}')

print('Training skeleton ready — uncomment above to train.')

## 💾 Cell 10 — Save dataset to Google Drive

In [ ]:
# If you generated to /content/hst_lensing_dataset and want to persist to Drive:
import shutil, os

SRC = '/content/hst_lensing_dataset'
DST = '/content/drive/MyDrive/hst_lensing_dataset'

# WARNING: this copies ~10-20GB. Only run once!
# shutil.copytree(SRC, DST, dirs_exist_ok=True)
# print(f'Saved to Google Drive: {DST}')

# Or zip the metadata + splits only (small):
!zip -r /content/drive/MyDrive/hst_metadata_splits.zip \
    {SRC}/metadata.csv \
    {SRC}/splits/ \
    {SRC}/dataset_config.json
print('Metadata zipped to Drive.')

## 📋 Cell 11 — Expected folder structure
```
hst_lensing_dataset/
├── metadata.csv                    ← master metadata (all 100k rows)
├── dataset_config.json             ← generation config snapshot
├── images/
│   ├── no_lens/
│   │   ├── clean/        img_*.png
│   │   ├── semi_messy/   img_*.png
│   │   └── messy/        img_*.png
│   ├── lens_no_subhalo/
│   │   ├── clean/
│   │   ├── semi_messy/
│   │   └── messy/
│   └── lens_with_subhalo/
│       ├── clean/
│       ├── semi_messy/
│       └── messy/
├── splits/
│   ├── train.csv                   ← 70% of all images
│   ├── val.csv                     ← 15%
│   ├── test.csv                    ← 15%
│   ├── model1_lens_vs_nolens.csv
│   ├── model2_lens_type.csv
│   └── model3_subhalo.csv
└── model_2_lens_type/
    ├── ring/             symlinks → images/
    ├── arc/
    ├── double/
    ├── quad/
    └── partial_ring/
```

**metadata.csv columns:**
```
filename, idx, seed, main_class, lens_label, lens_type, subhalo_label,
realism, split,
fwhm_arcsec, lens_theta_E, lens_e1, lens_e2, lens_gamma1, lens_gamma2,
lens_cx, lens_cy, lens_R_sersic, lens_n_sersic, lens_amp,
src_bx, src_by, src_R_sersic, src_n_sersic, src_amp, src_e1, src_e2,
subhalo_mass, subhalo_conc, subhalo_Rs, subhalo_alpha_Rs, subhalo_cx, subhalo_cy,
exposure_time, sky_background, read_noise_sigma, flux_jitter,
n_bg_galaxies, n_stars, cr_fraction, vignette_strength
```
